In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

# UNION and UNION BY NAME 

In [0]:
data1 = [('1', 'Alexa'), ('2', 'Siri')]
schema1 = 'id STRING, name STRING'

df1 = spark.createDataFrame(data1, schema1)

In [0]:
df1.display()

In [0]:
data2 = [('3', 'Cortana'), ('4', 'Javis')]
schema2 = 'id STRING, name STRING'
df2 = spark.createDataFrame(data2, schema2)
df2.display()

### Union

In [0]:
df1.union(df2).display()

### Union Vs Union by name
The main difference is union combines all the rows regardless of their columnname

**Example**

In [0]:
data1 = [('Alexa', '1'), ('Siri', '2')]
schema1 = 'name STRING, id STRING'

df1 = spark.createDataFrame(data1, schema1)
df1.union(df2).display()

### Union by name

In [0]:
df1.unionByName(df2).display()

# String Functions

In [0]:
df = spark.read.csv('/Volumes/workspace/default/firstvolume/BigMart Sales.csv', header=True, inferSchema=True)

In [0]:
df.display()

### initCap()
The first letter of all the words would be changed to capital

In [0]:
df.select(initcap('Item_Type')).display()

### Upper

In [0]:
df.select(upper('Item_Type').alias('Upper_Item_Type')).display()

### Lower

In [0]:
df.select(lower('Item_Type')).display()

# Date Functions

### current_date()

In [0]:
df = df.withColumn('curr_date', current_date())

df.display()

### date_add()

In [0]:
df = df.withColumn('week_after', date_add('curr_date', 7))
df.display()

### date_sub()

In [0]:
df = df.withColumn('week_before', date_sub('curr_date', 7))
df.show() 

### Alternative for date_sub()

In [0]:
df = df.withColumn('week_before_alt', date_add('curr_date', -7))
df.show()

### date_diff()
Find the difference or range between two dates

In [0]:
df = df.withColumn('datediff', datediff('week_after', 'curr_date'))
df.show()

In [0]:
df.withColumn('datediff', datediff('curr_date', 'week_after')).show()

### date_format()
Change the format of the date
Eg: dd-MM-yyyy -> yyyy-dd-MM

In [0]:
df = df.withColumn('week_before', date_format('week_before', 'dd-MM-yyyy'))
df.display()

# Handling NULL

### Dropping nulls

In [0]:
df.dropna(how='all').display()

In [0]:
df.dropna(how='any').display()

In [0]:
df.dropna(subset=["Item_Weight"]).display()

### Filling NULL values

In [0]:
df.fillna('NotAvailable').display()

#### Filling the Null values for a subset of columns

In [0]:
df.fillna(0, subset=["Item_Weight"]).display()

# Splitting and indexing

### Split

In [0]:
df.withColumn('Outlet_Type', split('Outlet_Type', ' ')).display()

### Index

In [0]:
df.withColumn('Outlet_Type', split('Outlet_Type', ' ')[1]).display()

### Explode
Explode splits the split list into multiple rows with one single value of the list

In [0]:
df_explode = df.withColumn('Outlet_Type', split('Outlet_Type', ' '))
df_explode.display()

In [0]:
df_explode.withColumn('Outlet_Type', explode('Outlet_type')).display()

### array_contains()
To check if a value is present in the list

In [0]:
df_explode.withColumn('Type1_flag', array_contains('Outlet_Type', 'Type1')).display()

# Group By

In [0]:
df.groupBy("Item_Type").sum("Item_MRP").display()

In [0]:
df.groupBy("Item_Type").agg(sum("Item_MRP")).display()

In [0]:
df.groupBy("Item_Type").agg(sum("Item_MRP"), avg("Item_MRP")).display()

In [0]:
# groupby 2 columns
df.groupBy("Item_Type", "Outlet_Size").sum("Item_MRP").display()

In [0]:
df.orderBy("Item_Type").groupBy("Item_Type", "Outlet_Size").agg(sum("Item_MRP"), avg("Item_MRP")).display()

### collect_list()
Group/Aggregate all the values for the same grouped value

In [0]:
data_books = [('user1', 'book1'), ('user1', 'book3'), 
              ('user2', 'book2'), ('user2', 'book3'), ('user3', 'book4')]
schema_books = 'user STRING, book STRING'
df_books = spark.createDataFrame(data_books, schema_books)
df_books.display()

In [0]:
df_books.groupBy("user").agg(collect_list("book")).display()